# 04 — 2x2 evaluation (Part 10)
A (M0, repair off) / B (M0, repair on) / C (M1, repair off) / D (M1, repair on). n=20 samples/problem, 3 seeds, unbiased pass@k, bootstrap CIs, per-category delta table, repair diagnostics, per-GPU-hour efficiency.

In [ ]:

# --- Self-contained Colab bootstrap (Part 0) ---
# Every notebook does this independently: Colab does not guarantee a new
# notebook tab reuses a previous notebook's VM, so nothing installed or
# cloned in another notebook can be assumed to exist here. This is
# idempotent -- re-running it (e.g. because you ARE still on the same
# runtime) just no-ops the clone and re-pulls latest.
import os, subprocess
from google.colab import drive
drive.mount('/content/drive')

REPO = "https://github.com/saiswaroop25-pixel/verilog-slm"
if not os.path.exists('/content/verilog-slm'):
    !git clone {REPO} /content/verilog-slm
%cd /content/verilog-slm
!git pull

CKPT = '/content/drive/MyDrive/verilog-slm/checkpoints'
LOGS = '/content/drive/MyDrive/verilog-slm/logs'
os.makedirs(CKPT, exist_ok=True); os.makedirs(LOGS, exist_ok=True)
!ln -sfn {CKPT} artifacts_drive_ckpt
!ln -sfn {LOGS} artifacts_drive_logs

In [ ]:
# Pinned deps (Part 0 hygiene: Colab silently upgrades packages between sessions)
!pip install -q -r requirements.txt -r requirements-train.txt

In [ ]:
# requirements-train.txt deliberately doesn't pin torch (Colab ships one
# already matched to the VM's CUDA driver) -- log what's actually here
# instead, per Part 0's "pin every dependency" hygiene.
import torch
print('torch:', torch.__version__, '| CUDA:', torch.version.cuda, '| GPU available:', torch.cuda.is_available())

In [ ]:
import os
# RTL toolchain: iverilog is required (compile+simulate). yosys and verible
# are optional -- the harness soft-gates on them (see docs/industry_standards.md)
# but install them here so the synthesis/lint stages actually run instead of
# being recorded as 'skipped'.
!apt-get -qq update && apt-get -qq install -y iverilog yosys > /dev/null

# verible's release asset filename embeds a version string that changes
# every release (verible-v0.0-NNNN-gHASH-linux-static-x86_64.tar.gz), so a
# fixed "latest/download/<literal-name>" URL goes stale -- resolve the
# actual asset URL via the GitHub API instead. Chained as one shell
# command (not separate `!` lines) so the VERIBLE_URL variable survives
# across the pipe/curl/tar steps -- each `!` line is its own subprocess,
# so a bare shell variable assignment on its own line would silently be
# treated as Python and never reach bash at all.
!VERIBLE_URL=$(curl -s https://api.github.com/repos/chipsalliance/verible/releases/latest \
  | grep -o '"browser_download_url": *"[^"]*linux-static-x86_64.tar.gz"' \
  | head -1 | cut -d'"' -f4) && echo "resolved verible URL: $VERIBLE_URL" && \
  curl -sL "$VERIBLE_URL" -o /tmp/verible.tar.gz && \
  mkdir -p /opt/verible && tar -xzf /tmp/verible.tar.gz -C /opt/verible --strip-components=1

os.environ['PATH'] += ':/opt/verible/bin'
!iverilog -V | head -1
!yosys -V
!verible-verilog-lint --version

In [ ]:
# Record the GPU model at the start of every run -- required for the
# per-GPU-hour metric (Part 0 non-negotiable hygiene) to mean anything.
!nvidia-smi -L

In [ ]:
import json
m0_meta = json.load(open('artifacts/m0/run_meta.json'))
m1_meta = json.load(open('artifacts/m1/run_meta.json'))
print(f"M0: {m0_meta['train_gpu_hours']:.2f} GPU-h on {m0_meta['gpu']}")
print(f"M1: {m1_meta['train_gpu_hours']:.2f} GPU-h on {m1_meta['gpu']}")

In [ ]:
# Simulation is CPU-bound (Part 4/11) -- switch this notebook's runtime to
# CPU-only for this cell if you're only running verification, or accept the
# GPU sitting idle during the multiprocessing verify pool below (generation
# itself still needs the GPU).
!python -m src.eval.run_eval \
  --m0-adapter artifacts/m0/final --m1-adapter artifacts/m1/final \
  --eval-sets data/eval/verilogeval_v2.jsonl data/eval/rtllm_v2.jsonl \
  --n 20 --k-repair 3 --seeds 1337 2025 7 \
  --m0-gpu-hours {m0_meta['train_gpu_hours']} --m1-gpu-hours {m1_meta['train_gpu_hours']} \
  --out artifacts/eval_report.json

## Headline table: pass@1 / pass@5 with 95% CI, all 4 cells, averaged over seeds

In [ ]:
import numpy as np
report = json.load(open('artifacts/eval_report.json'))
cells = ['M0_repair_off', 'M0_repair_on', 'M1_repair_off', 'M1_repair_on']
rows = []
for cell in cells:
    p1 = np.array([s['cells'][cell]['pass@1']['mean'] for s in report['per_seed']])
    p5 = np.array([s['cells'][cell]['pass@5']['mean'] for s in report['per_seed']])
    rows.append({'cell': cell, 'pass@1_mean': p1.mean(), 'pass@1_std': p1.std(),
                 'pass@5_mean': p5.mean(), 'pass@5_std': p5.std()})
import pandas as pd
pd.DataFrame(rows)

## Effects: curriculum, repair, interaction (seed 0 shown; repeat per-seed and compare against std above)

In [ ]:
effects = [s['effects'] for s in report['per_seed']]
pd.DataFrame(effects)

## Statistical honesty check (Part 10.6)
If the curriculum effect's magnitude is inside the cross-seed std above, say so plainly -- a 2-4% effect at this scale is plausible and reportable, but only if distinguishable from noise.

In [ ]:
curriculum_effects = [e['curriculum_effect_C_minus_A'] for e in effects]
print(f"curriculum effect: mean={np.mean(curriculum_effects):.4f}, std={np.std(curriculum_effects):.4f}")
print("=> inside noise" if np.std(curriculum_effects) > abs(np.mean(curriculum_effects)) else "=> exceeds seed noise")

## Per-category delta table -- the core scientific claim (Part 10.3)
Uses seed[0]'s taxonomy tables from the repair-off cells (the cleanest comparison: M0 vs M1 with no repair-loop confound).

In [ ]:
from src.eval.metrics import per_category_delta_table
TARGETED = {'incomplete_sensitivity', 'missing_default_case'}
# taxonomy_table entries are aggregated, not per-generation records; rebuild
# per-generation-shaped rows from the stored table for the delta computation
def expand(table):
    rows = []
    for row in table:
        rows += [{'ok': False, 'error_label': row['label']}] * row['count']
    return rows
m0_rows = expand(report['per_seed'][0]['cells']['M0_repair_off']['taxonomy_table'])
m1_rows = expand(report['per_seed'][0]['cells']['M1_repair_off']['taxonomy_table'])
delta_table = per_category_delta_table(m0_rows, m1_rows, TARGETED)
pd.DataFrame(delta_table).sort_values('delta')

## Repair-loop diagnostics (Part 10.4)

In [ ]:
for cell in ['M0_repair_on', 'M1_repair_on']:
    diag = report['per_seed'][0]['cells'][cell]['repair_diagnostics']
    print(cell, '->', json.dumps(diag, indent=2))

## Efficiency: accuracy per GPU-hour (Part 10.5)

In [ ]:
print(json.dumps(report.get('efficiency', {}), indent=2))

## Catch-all share sanity check across cells (Part 4/13)

In [ ]:
for cell in cells:
    share = report['per_seed'][0]['cells'][cell]['catch_all_share']
    flag = ' <-- taxonomy not discriminating well' if share > 0.6 else ''
    print(f"{cell}: catch_all_share={share:.1%}{flag}")